In [ ]:
import inspect
from transformers import TrainingArguments

print(inspect.signature(TrainingArguments.__init__))


(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict, str, NoneType] = <factory>, warmup_ratio: float = 0.0, warmup_ste

In [ ]:
!pip install -qU transformers datasets


In [ ]:
!pip install --upgrade transformers datasets huggingface-hub accelerate soundfile librosa
!apt-get update && apt-get install -y ffmpeg


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [ ]:
import transformers
print(transformers.__version__)  # 최소 4.4 이상이어야 합니다.


4.52.4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.exists("/content/drive/MyDrive/labels.csv"), "labels.csv 경로를 확인하세요"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import snapshot_download

# 1) Repo 전체(메타+LFS) 로컬로 내려받기
local_dir = snapshot_download(
    repo_id="bigfish951/whisper-project",
    repo_type="dataset",
    use_auth_token=True    # 로그인 되어 있으면 True
)

# 2) labels.csv 읽어서 경로를 전부 로컬 경로로 바꿔주기
import os, pandas as pd
df = pd.read_csv(os.path.join(local_dir, "labels.csv"))
df['file_path'] = df['file_path'].apply(lambda fn: os.path.join(local_dir, fn))

# 3) pandas → HuggingFace Dataset
from datasets import Dataset, Audio
ds = Dataset.from_pandas(df)

# 4) Audio 타입으로 캐스팅
ds = ds.cast_column("file_path", Audio(sampling_rate=16000))

# 5) train/test 분할
split_ds = ds.train_test_split(test_size=0.1)
train_ds = split_ds["train"]
eval_ds  = split_ds["test"]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 3003 files:   0%|          | 0/3003 [00:00<?, ?it/s]

In [ ]:
import os, pandas as pd
from huggingface_hub import snapshot_download

# (1) Dataset 전체 내려받기
local_dir = snapshot_download("bigfish951/whisper-project", repo_type="dataset", use_auth_token=True)

# (2) CSV 로드
csv_path = os.path.join(local_dir, "labels.csv")
df = pd.read_csv(csv_path)

# (3) URL에서 파일 이름만 추출
#     ex: 'https://.../dialog_00300480.wav' -> 'dialog_00300480.wav'
df['file_name'] = df['file_path'].apply(lambda url: url.split('/')[-1])

# (4) local_dir과 결합해서 진짜 파일 경로로 바꿔 주기
df['file_path'] = df['file_name'].apply(lambda fn: os.path.join(local_dir, fn))

# (5) 실제 존재하는 파일만 필터링
df = df[df['file_path'].apply(os.path.isfile)]
print("Filtered DataFrame shape:", df.shape)  # 이제 (3000, 3) 이 되어야 합니다.

# (6) 이제 HuggingFace Dataset으로 변환
from datasets import Dataset, Audio
ds = Dataset.from_pandas(df)
ds = ds.cast_column("file_path", Audio(sampling_rate=16000))

# (7) train/test split
split = ds.train_test_split(test_size=0.1)
train_dataset, eval_dataset = split["train"], split["test"]


Fetching 3003 files:   0%|          | 0/3003 [00:00<?, ?it/s]

Filtered DataFrame shape: (1500, 3)


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.feature_extractor,  # 혹은 processor.tokenizer
    model=   model,
    padding="longest",
)


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./whisper-output",
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 2,
    num_train_epochs            = 3,

    eval_strategy    = "steps",
    eval_steps       = 500,
    logging_strategy = "steps",
    logging_steps    = 100,
    save_strategy    = "steps",
    save_steps       = 500,

    predict_with_generate = True,
    generation_max_length = 128,
    generation_num_beams  = 4,

    learning_rate = 1e-5,
    warmup_steps  = 200,
    fp16          = torch.cuda.is_available(),
    report_to     = [],
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = eval_dataset,
    processing_class = processor,      # FeatureExtractor+Tokenizer
    data_collator    = data_collator,  # ⇐ 여기
)

trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
500,0.455800,0.233149


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=507, training_loss=2.2022367591218366, metrics={'train_runtime': 1284.1648, 'train_samples_per_second': 3.154, 'train_steps_per_second': 0.395, 'total_flos': 1.168770871296e+18, 'train_loss': 2.2022367591218366, 'epoch': 3.0})